# 情報論A 第5回：Tomasi-Kanade のSfM (2視点バージョン)

SIFTで自動対応付けを行い、`cv2.findFundamentalMat(..., cv2.FM_RANSAC)` で外れ値を除去した対応点だけを用いて、Tomasi-Kanadeの正射影SfMを実装します。

画像は `sfm_ref.jpg` と `sfm_src.jpg` を使います。

若干埋まっていない部分があるので、探して埋めて完成させてください

### 準備（変更不要）

`samples`に入っている `sfm_ref.jpg` と `sfm_src.jpg` をColabにアップロードしてください。

Google Driveを使う場合は、次のセルのコメントアウトを外してマウントしてください。

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

# Googleドライブをマウントする場合
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/My Drive/Colab Notebooks/johoronA/"


In [ ]:
# 画像の読み込み
ref = cv2.imread("sfm_ref.jpg") # 1枚目の画像（BGR）
src = cv2.imread("sfm_src.jpg") # 2枚目の画像（BGR）

# SIFTによる特徴点検出と特徴量記述
sift = cv2.SIFT_create()
kp_ref, des_ref = sift.detectAndCompute(ref, None)
kp_src, des_src = sift.detectAndCompute(src, None)

# SIFT特徴量はL2距離でマッチングする
matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
matches = matcher.match(des_ref, des_src)

# 距離が小さい順に並べる
matches = sorted(matches, key=lambda x: x.distance)
print("# raw matches:", len(matches))

# マッチング結果を可視化
corr_disp = cv2.drawMatches(
    ref, kp_ref, src, kp_src, matches[:100], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)
cv2_imshow(corr_disp)

# 対応点の登録
# ref_points[p] と src_points[p] が同じ3D点に対応するとみなす
dst_points = np.float32([kp_ref[m.queryIdx].pt for m in matches]).reshape(-1, 2)
src_points = np.float32([kp_src[m.trainIdx].pt for m in matches]).reshape(-1, 2)


## RANSACによる外れ値除去（変更不要）

ここでは、OpenCVの `cv2.findFundamentalMat` と RANSAC を使い、ホモグラフィ推定でinlierと判定された対応点だけを残します。

※ここでFundamental Matrix推定をやってしまうのはなんか順序おかしいんですが許してください

In [ ]:
# ============================================================
# RANSACによる外れ値除去
#   - 今回は、HomographyではなくFundamental matrixを使う
#   - src_points, dst_points は前のセルで作成済みのものを使う
#   - src_points: src画像上の点
#   - dst_points: ref画像上の対応点
# ============================================================

# RANSACのパラメータ
threshold = 2.0  # 厳しければ 3.0 や 5.0 に緩める

# Fundamental matrix を RANSACで推定
# 対応関係は src_points -> dst_points として入れる
F, mask = cv2.findFundamentalMat(
    src_points,
    dst_points,
    method=cv2.FM_RANSAC,
    ransacReprojThreshold=threshold,
    confidence=0.99
)

if F is None or mask is None:
    raise RuntimeError("Fundamental matrix estimation failed. Try increasing threshold.")

# inlier maskを作る
best_inliers = mask.ravel().astype(bool)
best_num_inliers = np.sum(best_inliers)

print("inliers:", best_num_inliers, "/", len(src_points))
print("inlier ratio:", best_num_inliers / len(src_points))
print("F =")
print(F)

# inlierだけ残した対応点
src_points_in = src_points[best_inliers]
dst_points_in = dst_points[best_inliers]

print("src_points_in:", src_points_in.shape)
print("dst_points_in:", dst_points_in.shape)

# inlier対応だけを可視化
matches_in = [
    m for m, keep in zip(matches, best_inliers)
    if keep
]

corr_disp_in = cv2.drawMatches(
    ref, kp_ref,
    src, kp_src,
    matches_in[:100],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

cv2_imshow(corr_disp_in)

# Tomasi-Kanade以降のセルで使う変数名に合わせる
src_in = src_points_in
ref_in = dst_points_in

print("ref_in:", ref_in.shape)
print("src_in:", src_in.shape)

## Tomasi-Kanade の2-view factorization

正射影を仮定すると、各フレームの画像座標は

$$
u_{fp} = \mathbf{i}_f^\top \mathbf{x}_p, \qquad
v_{fp} = \mathbf{j}_f^\top \mathbf{x}_p
$$

と書けます。したがって、各フレームで重心を引いた観測行列は

$$
W = MS, \qquad \operatorname{rank}(W) \leq 3
$$

になります。ここでは2枚の画像なので、観測行列は $4 \times P$ です。

In [ ]:
def _metric_coeff(a, b):
    """Return coefficients of a^T G b for symmetric 3x3 G.

    G is represented by
    [g11, g12, g13, g22, g23, g33].
    """
    ax, ay, az = a
    bx, by, bz = b
    return np.array([
        ax * bx,
        ax * by + ay * bx,
        ax * bz + az * bx,
        ay * by,
        ay * bz + az * by,
        az * bz,
    ], dtype=np.float64)


def tomasi_kanade_two_view(ref_points, src_points, align_first_view=True):
    """Tomasi--Kanade型の2-view SfMを行う。

    Parameters
    ----------
    ref_points : ndarray, shape (P, 2)
        1枚目画像上の対応点。
    src_points : ndarray, shape (P, 2)
        2枚目画像上の対応点。
    align_first_view : bool
        Trueなら、1枚目のカメラ軸がx-y平面に揃うように全体を回転する。

    Returns
    -------
    result : dict
        W, W_centered, M, S, row_means などを含む辞書。
    """
    if len(ref_points) != len(src_points):
        raise ValueError("ref_points と src_points の点数が一致していません。")
    if len(ref_points) < 4:
        raise ValueError("少なくとも4点以上の対応点が必要です。")

    # 観測行列 W_raw を作る。2 viewなので 4 x P。
    W_raw = np.vstack([
        ref_points[:, 0],
        ref_points[:, 1],
        src_points[:, 0],
        src_points[:, 1],
    ]).astype(np.float64)

    # 各行の重心を引く。これにより画像平面内の並進成分を除去する。
    row_means = W_raw.mean(axis=1, keepdims=True)
    W = W_raw - row_means

    # SVD
    U, Sigma, Vt = np.linalg.svd(W, full_matrices=False)

    # 課題（ここを埋める）：上位3つの特異値に対応する部分を取り出す
    U3 = ...
    Sigma3 = ...
    V3t = ...

    sqrt_Sigma3 = np.diag(np.sqrt(Sigma3))
    M_hat = U3 @ sqrt_Sigma3
    S_hat = sqrt_Sigma3 @ V3t

    # SVD分解には線形曖昧性がある。
    # M = M_hat A, S = A^{-1} S_hat となる A を、metric constraintsから求める。
    equations = []
    values = []

    for f in range(2):
        ihat = M_hat[2 * f, :]
        jhat = M_hat[2 * f + 1, :]

        # ||i_f||^2 = 1
        equations.append(_metric_coeff(ihat, ihat))
        values.append(1.0)

        # ||j_f||^2 = 1
        equations.append(_metric_coeff(jhat, jhat))
        values.append(1.0)

        # i_f^T j_f = 0
        equations.append(_metric_coeff(ihat, jhat))
        values.append(0.0)

    A_eq = np.vstack(equations)
    b_eq = np.array(values)

    # G = A A^T を最小二乗で求める
    g, residuals, rank, svals = np.linalg.lstsq(A_eq, b_eq, rcond=None)
    G = np.array([
        [g[0], g[1], g[2]],
        [g[1], g[3], g[4]],
        [g[2], g[4], g[5]],
    ], dtype=np.float64)
    G = 0.5 * (G + G.T)

    # 数値誤差でGがわずかに非正定値になる場合に備え、固有値をクリップする
    evals, evecs = np.linalg.eigh(G)
    if np.any(evals <= 0):
        print("Warning: G has non-positive eigenvalues. Clipping them for numerical stability.")
        print("eigenvalues before clipping:", evals)
    evals_clipped = np.clip(evals, 1e-8, None)
    A_metric = evecs @ np.diag(np.sqrt(evals_clipped))

    # metric upgrade
    M = M_hat @ A_metric
    S = np.linalg.pinv(A_metric) @ S_hat

    # 1枚目の視点に座標系を揃える。可視化が少し見やすくなる。
    if align_first_view:
        i1 = M[0, :]
        j1 = M[1, :]
        i1 = i1 / np.linalg.norm(i1)
        j1 = j1 - np.dot(i1, j1) * i1
        j1 = j1 / np.linalg.norm(j1)
        k1 = np.cross(i1, j1)
        k1 = k1 / np.linalg.norm(k1)
        O = np.vstack([i1, j1, k1])
        M = M @ O.T
        S = O @ S

    W_reproj = M @ S
    reproj_error = np.sqrt(np.mean((W - W_reproj) ** 2))

    return {
        "W_raw": W_raw,
        "W": W,
        "row_means": row_means,
        "M_hat": M_hat,
        "S_hat": S_hat,
        "G": G,
        "A_metric": A_metric,
        "M": M,
        "S": S,
        "singular_values": Sigma,
        "W_reproj": W_reproj,
        "reproj_error": reproj_error,
    }


reprojection errorが数画素程度になっていれば多分OK

In [ ]:
# Tomasi--Kanade 2-view SfMを実行
result = tomasi_kanade_two_view(ref_in, src_in)

W = result["W"]
M = result["M"]
S = result["S"]

print("singular values of W:")
print(result["singular_values"])
print()

print("W shape:", W.shape)
print("M shape:", M.shape)
print("S shape:", S.shape)
print("RMS reprojection error on centered coordinates:", result["reproj_error"])
print()

# metric constraints の確認
for f in range(2):
    i = M[2 * f, :]
    j = M[2 * f + 1, :]
    print(f"frame {f+1}")
    print("  ||i|| =", np.linalg.norm(i))
    print("  ||j|| =", np.linalg.norm(j))
    print("  i dot j =", np.dot(i, j))


## 復元された3D点群の可視化（変更不要）

Tomasi-Kanade法では、絶対的なスケールや座標系の向きは任意です。そのため、ここでは形状の相対的な配置を3D散布図として表示します。

In [ ]:
# ============================================================
# 復元された3D点群の可視化
#   - 現在の実装では、復元点群は result["S"] または S に入っている
#   - S は shape (3, N) の shape matrix
#   - 外れ値に表示範囲が引っ張られないように、
#     中央 keep_ratio 分の点が入る範囲にズームして表示する
# ============================================================

def set_robust_3d_limits(ax, X, keep_ratio=0.90, equal_aspect=True):
    """
    3D点群の表示範囲をロバストに設定する関数。

    Parameters
    ----------
    ax : matplotlib 3D axis
        表示対象の3D axis。
    X : ndarray, shape (N, 3)
        3D点群。
    keep_ratio : float
        表示範囲に入れたい中央部分の割合。
        0.90なら各軸ごとに中央90%が入る範囲を表示する。
    equal_aspect : bool
        Trueなら3軸の表示スケールを揃える。
    """
    assert X.ndim == 2 and X.shape[1] == 3

    q_low = (1.0 - keep_ratio) / 2.0
    q_high = 1.0 - q_low

    mins = np.quantile(X, q_low, axis=0)
    maxs = np.quantile(X, q_high, axis=0)

    if equal_aspect:
        center = 0.5 * (mins + maxs)
        radius = 0.5 * np.max(maxs - mins)

        # 範囲が極端に小さい場合の保険
        if radius <= 1e-12:
            radius = 1.0

        ax.set_xlim(center[0] - radius, center[0] + radius)
        ax.set_ylim(center[1] - radius, center[1] + radius)
        ax.set_zlim(center[2] - radius, center[2] + radius)
    else:
        ax.set_xlim(mins[0], maxs[0])
        ax.set_ylim(mins[1], maxs[1])
        ax.set_zlim(mins[2], maxs[2])


# ------------------------------------------------------------
# 現在の実装に合わせて3D点群を取得
# ------------------------------------------------------------
if "result" in globals() and isinstance(result, dict) and "S" in result:
    S_use = result["S"]
elif "S" in globals():
    S_use = S
else:
    raise NameError("3D点群が見つかりません。先に Tomasi-Kanade SfM のセルを実行してください。")

S_use = np.asarray(S_use)

# S は通常 shape (3, N)
# 可視化用には shape (N, 3) にする
if S_use.shape[0] == 3:
    X_vis = S_use.T
elif S_use.shape[1] == 3:
    X_vis = S_use
else:
    raise ValueError(f"S のshapeが想定外です: {S_use.shape}. shape (3, N) または (N, 3) を想定しています。")

# NaNやInfが混ざっていた場合は除外
valid = np.isfinite(X_vis).all(axis=1)
X_vis = X_vis[valid]

print("Number of valid 3D points:", len(X_vis))

# 表示範囲に使う点の割合
# 0.90: 中央90%にズーム
# 0.80: より強くズーム
# 0.95: やや広め
keep_ratio = 0.90

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    X_vis[:, 0],
    X_vis[:, 1],
    X_vis[:, 2],
    s=4,
    alpha=0.8
)

set_robust_3d_limits(
    ax,
    X_vis,
    keep_ratio=keep_ratio,
    equal_aspect=True
)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title(f"Reconstructed 3D points (view range: central {int(keep_ratio * 100)}%)")

# 見やすい角度。必要に応じて変更
ax.view_init(elev=20, azim=60)

plt.show()

## 再投影の確認（変更不要）

復元された $M$ と $S$ から、重心を引いた画像座標を再投影し、元の対応点と重ねて確認します。

In [ ]:
W_reproj = result["W_reproj"]
row_means = result["row_means"]
W_reproj_raw = W_reproj + row_means

ref_reproj = np.vstack([W_reproj_raw[0], W_reproj_raw[1]]).T
src_reproj = np.vstack([W_reproj_raw[2], W_reproj_raw[3]]).T

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(ref, cv2.COLOR_BGR2RGB))
plt.scatter(ref_in[:, 0], ref_in[:, 1], s=8, label='observed')
plt.scatter(ref_reproj[:, 0], ref_reproj[:, 1], s=8, marker='x', label='reprojected')
plt.title('ref image')
plt.legend()
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(src, cv2.COLOR_BGR2RGB))
plt.scatter(src_in[:, 0], src_in[:, 1], s=8, label='observed')
plt.scatter(src_reproj[:, 0], src_reproj[:, 1], s=8, marker='x', label='reprojected')
plt.title('src image')
plt.legend()
plt.axis('off')

plt.show()
